## Import necessary modules


In [208]:
import pandas as pd
import numpy as np
from FinMind.data import DataLoader
from pandas import api
import requests
import time
from pprint import pprint

## Position Cost Distribution (PCD) Analysis

This function estimates the **Position Cost Distribution (PCD)** of all outstanding shares to help identify support/resistance levels and market profitability.

**🧠 How it Works:**
It divides the historical price range into discrete buckets (bins). For each K-line, it simulates market turnover: existing holdings decay based on the daily volume, while new volume is accumulated evenly across the day's high-low price range.

**📥 Inputs:**

- `df`: Market data (requires `max`, `min`, `close`, `Trading_Volume`, `TotalShares`).
- `num_buckets`: Resolution of the price bins (default: `400`).
- `total_shares`: Total outstanding shares.

**📤 Key Outputs:**

- **Current & Average Price:** Latest close vs. volume-weighted average cost.
- **Profit Ratio (%):** Percentage of outstanding shares currently held at a profit.
- **90% & 70% Cost Ranges:** The price intervals containing 90% and 70% of the distribution.
- **Range Overlap (%):** Ratio of the 70% range width to the 90% range width (lower % = higher chip concentration).


In [209]:
def calculate_position_cost_distribution(df: pd.DataFrame, num_buckets: int = 400, total_shares: float = None) -> dict:
       """
       計算部位成本分佈 (Position Cost Distribution, PCD)
       
       參數:
              df: pd.DataFrame, 必須包含 'max', 'min', 'close', 'Trading_Volume', 'TotalShares' 欄位
              num_buckets: int, 將價格區間切分的數量 (對應 Pine Script 的 NUM_BUCKETS)
              total_shares: float, 總發行股數
       回傳:
              dict, 包含 PCD 統計指標與原始分佈資料
       """
       # 確保資料依時間遞增排序
       df = df.sort_index(ascending=True)
       
       # 取得歷史最高與最低價以定義價格範圍
       min_price = df['min'].min()
       max_price = df['max'].max()
       
       # 避免最高與最低價相同導致除以零
       if max_price == min_price:
              step = 0.01
       else:
              step = (max_price - min_price) / num_buckets
       # 初始化分佈陣列與對應的價格標籤
       dist = np.zeros(num_buckets)
       bucket_prices = np.array([(i + 0.5) * step + min_price for i in range(num_buckets)])
       # 輔助函式：取得價格對應的 bucket 索引
       def get_bucket_index(price):
              # 防呆處理，避免索引超出範圍
              idx = int(np.floor((price - min_price) / step))
              return max(0, min(idx, num_buckets - 1))
       is_first_candle = True
       # 逐K線模擬籌碼換手
       for _, row in df.iterrows():
              if pd.isna(total_shares) or total_shares <= 0:
                     continue
              # 計算換手率
              turnover = row['Trading_Volume'] / total_shares
              
              # 取得高低點跨越的 bucket 索引
              start_idx = get_bucket_index(row['min'])
              end_idx = get_bucket_index(row['max'])
              buckets_spanned = end_idx - start_idx + 1

              if is_first_candle:
                     # 第一根 K 線：將所有發行股數均勻分佈在當天的高低點區間
                     shares_per_bucket = total_shares / buckets_spanned
                     dist[start_idx:end_idx + 1] = shares_per_bucket
                     is_first_candle = False
              else:
                     # 後續 K 線：
                     # 1. 舊籌碼根據換手率衰減
                     dist *= (1 - turnover)
                     # 2. 新成交量均勻加入到當天的高低點區間
                     shares_per_bucket = row['Trading_Volume'] / buckets_spanned
                     dist[start_idx:end_idx + 1] += shares_per_bucket
       # --- 計算統計指標 ---
       
       # 計算累積分配 (Cumulative Distribution)
       cumdist = np.cumsum(dist)
       total_dist_shares = cumdist[-1]
       
       if total_dist_shares == 0:
              return None # 避免無有效數據

       # 目前價格
       current_price = df['close'].iloc[-1]
       close_index = get_bucket_index(current_price)

       # 獲利比例 (Profit Ratio)：低於或等於當前價格的籌碼比例
       profit_index = min(close_index + 1, num_buckets - 1)
       profit_ratio = cumdist[profit_index] / total_dist_shares

       # 平均持倉成本 (Average Price)
       avg_price = np.sum(bucket_prices * (dist / total_dist_shares))

       # 尋找指定百分比對應的價格區間 (等同 Pine Script binary_search_leftmost)
       p05_idx = np.searchsorted(cumdist, total_dist_shares * 0.05)
       p95_idx = np.searchsorted(cumdist, total_dist_shares * 0.95)
       p15_idx = np.searchsorted(cumdist, total_dist_shares * 0.15)
       p85_idx = np.searchsorted(cumdist, total_dist_shares * 0.85)

       ninety_pct_low = bucket_prices[min(p05_idx, num_buckets - 1)]
       ninety_pct_high = bucket_prices[min(p95_idx, num_buckets - 1)]
       seventy_pct_low = bucket_prices[min(p15_idx, num_buckets - 1)]
       seventy_pct_high = bucket_prices[min(p85_idx, num_buckets - 1)]

       # 區間重疊度 (Range Overlap)
       range_overlap = 0.0
       if ninety_pct_high != ninety_pct_low:
              range_overlap = (seventy_pct_high - seventy_pct_low) / (ninety_pct_high - ninety_pct_low)

       return {
              'Current Price': current_price,
              'Average Price': avg_price,
              'Profit Ratio (%)': profit_ratio * 100,
              '90% Cost Range': (ninety_pct_low, ninety_pct_high),
              '70% Cost Range': (seventy_pct_low, seventy_pct_high),
              'Range Overlap (%)': range_overlap * 100,
              'Raw Data': {
              'prices': bucket_prices,
              'distribution': dist
              }
       }

## Data Retrieval for PCD

This function fetches the prerequisite market data and outstanding share counts required to calculate the Position Cost Distribution (PCD).

**🧠 How it Works:**
It utilizes a `DataLoader` to pull two distinct datasets for a given stock: the latest shareholding records (to extract the total number of issued shares) and the daily historical price/volume data starting from a specified date.

**📥 Inputs:**

- `stock_id` (str): The target stock ticker (e.g., `'2330'`).
- `start_date` (str): The starting date for historical daily data (default: `'2024-01-01'`).

**📤 Outputs:**
Returns a tuple containing:

- `data` (`pd.DataFrame`): Historical daily market data (including high, low, close, and volume).
- `total_shares` (`float`): The most recent total number of issued shares.


In [210]:
def get_stock_data_for_PCD(stock_id: str, start_date: str = '2024-01-01') -> tuple[pd.DataFrame, float]:
       """
       取得股票歷史價格與總發行股數資料，供計算部位成本分佈使用
       參數:
              stock_id: str, 股票代碼 (例如 '2330')
              start_date: str, 資料起始日期 (格式 'YYYY-MM-DD')
       回傳:
              tuple: (歷史價格 DataFrame, 總發行股數)
       
       """
       dl = DataLoader()
       # 2. Fetch the Foreign Shareholding data
       df = dl.taiwan_stock_shareholding(
              stock_id=stock_id,
              start_date="2024-04-01",
       )
       latest_record = df.iloc[-1]
       total_shares = latest_record['NumberOfSharesIssued']
       data = dl.taiwan_stock_daily(stock_id=stock_id, start_date=start_date)
       return data, total_shares

## TWSE Limit-Up Stock Screener

This function fetches daily trading data directly from the Taiwan Stock Exchange (TWSE) API to identify stocks that hit the daily upper circuit limit (漲停板).

**🧠 How it Works:**
It requests the comprehensive daily closing report (`MI_INDEX`) for all equities. Since the raw data formats the price change as an absolute value with a separate +/- sign, the function reconstructs the previous day's closing price, calculates the exact percentage gain, and filters the dataset for stocks surging over 9.0%.

**📥 Inputs:**

- `date_str` (`str`): The target trading date formatted as `YYYYMMDD` (e.g., `'20240412'`).

**📤 Outputs:**

- Returns a `pd.DataFrame` containing the filtered limit-up stocks, complete with parsed numerical prices, the calculated previous close, and the exact daily percentage gain (`漲幅(%)`). If the market was closed or the API fails, it returns an empty DataFrame.


In [211]:
def get_twse_data(date_str):
       """
       獲取 TWSE 當天所有漲幅超過 9% 的股票
       :param date_str: 格式為 'YYYYMMDD'，例如 '20240412'
       """
       # type=ALLBUT0999 代表「全部(不含權證、牛熊證、可展延牛熊證)」
       url = f"https://www.twse.com.tw/exchangeReport/MI_INDEX?response=json&date={date_str}&type=ALLBUT0999"
       
       # 必須加入 User-Agent，否則會被 TWSE 阻擋
       headers = {
              "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
       }
       
       try:
              res = requests.get(url, headers=headers)
              data = res.json()
       except Exception as e:
              print(f"API 請求失敗: {e}")
              return pd.DataFrame()

       if data.get('stat') != 'OK':
              print(f"[{date_str}] 無資料或為休市日")
              return pd.DataFrame()
       df = pd.DataFrame(data['tables'][-2]['data'], columns=data['tables'][-2]['fields'])

       # 1. 動態尋找包含「收盤價」的正確表格 (迴避 data9/data8 變動的問題)
       def parse_sign(x):
              x_str = str(x)
              if '+' in x_str: return 1
              elif '-' in x_str: return -1
              else: return 0
       df['sign'] = df['漲跌(+/-)'].apply(parse_sign)
       # 昨收 = 今收 - (漲跌價差 * 符號)
       df['收盤價'] = df['收盤價'].replace('--', np.nan)
       df['收盤價'] = df['收盤價'].str.replace(',', '', regex=False).astype(float)
       df['漲跌價差'] = df['漲跌價差'].str.replace(',', '', regex=False).astype(float)
       df['prev_close'] = df['收盤價'] - (df['漲跌價差'] * df['sign'])
       
       # 計算漲幅百分比 (%)
       df['漲幅(%)'] = (df['收盤價'] - df['prev_close']) / df['prev_close'] * 100

       # 5. 篩選漲幅超過 9% 的股票
       df_over_9 = df[df['漲幅(%)'] > 9.0].copy()
       return df_over_9

In [222]:
def get_tpex_data(date_str):
       """取得 TPEx (上櫃) 收盤行情"""
       # 將 YYYYMMDD 轉為 民國年 YYY/MM/DD
       tw_year = int(date_str[:4]) - 1911
       tpex_date = f"{tw_year}/{date_str[4:6]}/{date_str[6:8]}"
       
       # se=AL 代表全部上櫃證券
       url = f"https://www.tpex.org.tw/web/stock/aftertrading/otc_quotes_no1430/stk_wn1430_result.php?l=zh-tw&d={tpex_date}&se=AL"
       headers = {"User-Agent": "Mozilla/5.0"}
       
       try:
              res = requests.get(url, headers=headers)
              data = res.json()
       except Exception as e:
              print(f"TPEx API 失敗: {e}")
              return pd.DataFrame()
       df = pd.DataFrame(data['tables'][0]['data'], columns=data['tables'][0]['fields'])
       # 將代號轉為字串後，只保留長度剛好為 4 的列
       df = df[df['代號'].astype(str).str.len() == 4].copy() 
       # TPEx 的漲跌欄位直接包含符號與數字 (例如 '+1.50', '-0.20', '0.00')
       # 遇到沒有漲跌的股票可能會顯示 'X0.00' 或空格
       df['漲跌'] = df['漲跌'].str.replace('X', '', regex=False)
       # print(df.head())
       # print(df.columns)
       def extract_sign(x):
              if '+' in str(x): return 1
              elif '-' in str(x): return -1
              else: return 0
              
       def extract_spread(x):
              # 移除正負號，只保留數字部分
              val = str(x).replace('+', '').replace('-', '').strip()
              return val if val else '0'

       df['sign'] = df['漲跌'].apply(extract_sign)
       df['漲跌價差'] = df['漲跌'].apply(extract_spread)
       df['收盤 '] = df['收盤 '].replace('----', np.nan)
       df['收盤 '] = df['收盤 '].replace('除息', np.nan, regex=False)
       df['收盤 '] = df['收盤 '].replace('除權', np.nan, regex=False)
       df['收盤 '] = df['收盤 '].str.replace(',', '', regex=False).astype(float)
       df['漲跌價差'] = df['漲跌價差'].replace('除息', np.nan, regex=False)
       df['漲跌價差'] = df['漲跌價差'].replace('除權', np.nan, regex=False)
       df['漲跌價差'] = df['漲跌價差'].replace('----', np.nan, regex=False)
       df['previous_close'] = df['收盤 '] - (df['漲跌價差'].str.replace(',', '', regex=False).astype(float) * df['sign'])
       df['漲幅(%)'] = (df['收盤 '] - df['previous_close']) / df['previous_close'] * 100
       
       df_over_9 = df[df['漲幅(%)'] > 9.0].copy()
       return df_over_9

       # df = get_tpex_data('20260415')


In [223]:
## main code


if __name__ == "__main__":
       # 1. 取得 TWSE 當天漲停的股票
       today_str = time.strftime("%Y%m%d")
       today_str = '20260416' # 測試用，固定日期
       df_over9 = get_twse_data(today_str)
       # print(f"\n{today_str} 共有 {len(df_over9)} 檔上市股票漲停：")
       # print(df_over9[['證券代號', '證券名稱', '收盤價', '漲幅(%)']])
       df_tpex_over9 = get_tpex_data(today_str)
       # print(f"\n{today_str} 共有 {len(df_tpex_over9)} 檔上櫃股票漲停：")
       # print(df_tpex_over9[['代號', '名稱', '收盤 ', '漲幅(%)']])
       id_over_9 = set(df_over9['證券代號'].astype(str)).union(set(df_tpex_over9['代號'].astype(str)))
       print(f"\n{today_str} 上市+上櫃共有 {len(id_over_9)} 檔股票漲停：")
       print(id_over_9)
       for stock_id in id_over_9: 
              print(f"\n=== {stock_id} 的部位成本分佈 ===")
              try:
                     data, total_shares = get_stock_data_for_PCD(stock_id, start_date='2024-01-01')
                     pcd_result = calculate_position_cost_distribution(data, total_shares=total_shares)
                     if pcd_result is not None and (pcd_result['70% Cost Range'][1] - pcd_result['70% Cost Range'][0] ) / pcd_result['Current Price'] * 100 < 15:
                            print(f"目前價格: {pcd_result['Current Price']:.2f}")
                            print(f"平均持倉成本: {pcd_result['Average Price']:.2f}")
                            print(f"獲利比例: {pcd_result['Profit Ratio (%)']:.2f}%")
                            print(f"90% Cost Range:  {pcd_result['90% Cost Range'][0]:.2f} - {pcd_result['90% Cost Range'][1]:.2f}")
                            print(f"90% Concentration:  { (pcd_result['90% Cost Range'][1] - pcd_result['90% Cost Range'][0] ) / pcd_result['Current Price'] * 100:.2f}%")
                            print(f"70% Cost Range:  {pcd_result['70% Cost Range'][0]:.2f} - {pcd_result['70% Cost Range'][1]:.2f}")
                            print(f"70% Concentration:  { (pcd_result['70% Cost Range'][1] - pcd_result['70% Cost Range'][0] ) / pcd_result['Current Price'] * 100:.2f}%")
                            print(f"區間重疊度: {pcd_result['Range Overlap (%)']:.2f}%")
                     else:
                            print("70% Cost Range 超過 15%，不列印詳細資訊")
              except Exception as e:
                     print(f"處理 {stock_id} 時發生錯誤: {e}")
       


2026-04-17 00:17:34.861 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockShareholding, data_id: 3518



20260416 上市+上櫃共有 76 檔股票漲停：
{'3518', '1626', '3054', '3587', '6218', '8121', '7712', '3094', '7631', '4979', '1595', '6953', '8182', '5011', '5228', '3272', '3229', '4741', '2484', '6419', '4905', '2303', '4919', '6285', '8092', '6173', '6877', '6829', '3105', '6462', '2340', '6890', '6806', '8046', '3581', '3388', '6174', '4977', '2312', '6902', '6684', '3555', '8227', '3339', '6530', '2923', '6141', '8150', '6208', '8043', '6456', '6417', '9105', '8040', '6213', '6163', '3219', '4960', '5498', '3221', '2363', '8289', '4908', '3529', '5269', '6415', '6657', '6533', '4939', '4556', '3714', '5340', '4714', '6441', '3576', '3049'}

=== 3518 的部位成本分佈 ===


2026-04-17 00:17:42.446 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 3518
2026-04-17 00:17:42.573 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockShareholding, data_id: 1626


70% Cost Range 超過 15%，不列印詳細資訊

=== 1626 的部位成本分佈 ===


2026-04-17 00:17:43.834 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 1626
2026-04-17 00:17:43.926 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockShareholding, data_id: 3054


...

70% Cost Range 超過 15%，不列印詳細資訊

=== 3576 的部位成本分佈 ===


2026-04-17 00:19:22.173 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 3576
2026-04-17 00:19:22.286 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockShareholding, data_id: 3049


70% Cost Range 超過 15%，不列印詳細資訊

=== 3049 的部位成本分佈 ===


2026-04-17 00:19:23.266 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 3049


70% Cost Range 超過 15%，不列印詳細資訊
